In [0]:
df = spark.read.table("sales_forecast_project.bronze.prodcuts_data")

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW latest_products AS
SELECT * EXCEPT (rn) FROM (
SELECT *, 
ROW_NUMBER() OVER (
    PARTITION BY Product_ID
    ORDER BY ingestion_time DESC 
    ) AS rn
FROM sales_forecast_project.bronze.prodcuts_data
) WHERE rn = 1

In [0]:
%sql
CREATE TABLE IF NOT EXISTS sales_forecast_project.silver.products_data
AS
SELECT * FROM latest_products
WHERE 1=0;

In [0]:
%sql
MERGE INTO sales_forecast_project.silver.products_data ps
using latest_products pb
ON pb.Product_ID = ps.Product_ID

WHEN MATCHED THEN UPDATE SET *

WHEN NOT MATCHED THEN INSERT *;


In [0]:
%sql
SELECT * FROM sales_forecast_project.silver.products_data
WHERE Product_ID = "SKU051";